💡 **Environment:** `clamp-analyses`

# ARCHS4 CRISPR-Cas9 - Gene Enrichment Analysis

Scores every latent variable of the ARCHS4 canonical model against the CRISPR-Cas9 lipid screen gene sets and selects the ones significant at FDR < 0.05.


## Load results

In [ ]:
library(dplyr)
library(readr)
library(ggplot2)

csv_path <- snakemake@input[["fgsea"]]
fdr_threshold <- snakemake@params[["fdr"]]

df <- read.csv(csv_path, stringsAsFactors = FALSE)

cat("Rows (LV x gene-set x rep):", nrow(df), "\n")
cat("LVs scored:", dplyr::n_distinct(df$lv), "\n")
cat("Gene sets:", paste(unique(df$pathway), collapse = ", "), "\n")
head(df)


## Best p-value per LV

In [ ]:
lv_best <- df %>%
  dplyr::group_by(lv, pathway) %>%
  dplyr::summarise(best_padj = min(padj), best_NES = max(NES), .groups = "drop") %>%
  dplyr::arrange(best_padj)

cat("Top 20 LV x gene-set pairs by best padj:\n")
print(head(lv_best, 20))

cat("\nLVs with best_padj <", fdr_threshold, "for at least one gene set:",
    dplyr::n_distinct(lv_best$lv[lv_best$best_padj < fdr_threshold]), "of", dplyr::n_distinct(lv_best$lv), "\n")


In [ ]:
options(repr.plot.width = 8, repr.plot.height = 5)

ggplot(lv_best, aes(x = -log10(pmax(best_padj, .Machine$double.xmin)), fill = pathway)) +
  geom_histogram(bins = 60, alpha = 0.7, position = "identity") +
  geom_vline(xintercept = -log10(fdr_threshold), linetype = "dashed", colour = "grey30") +
  labs(
    title = "Best padj across all LVs",
    x = expression("-log"[10]*"(best padj across 10 reps)"), y = "LV count", fill = "CRISPR gene set"
  ) +
  theme_bw(base_size = 12)


## Select CRISPR-significant LVs

In [ ]:
lv_order <- lv_best %>%
  dplyr::filter(best_padj < fdr_threshold) %>%
  dplyr::group_by(lv) %>%
  dplyr::summarise(gsea_padj = min(best_padj), .groups = "drop") %>%
  dplyr::arrange(gsea_padj) %>%
  dplyr::mutate(lv_rank = dplyr::row_number())

cat("CRISPR-significant LVs (FDR <", fdr_threshold, "):", nrow(lv_order), "of", dplyr::n_distinct(lv_best$lv), "\n")
cat("Top 5 by rank:\n")
print(head(lv_order, 5))

readr::write_csv(lv_order, snakemake@output[["lv_sig"]])
cat("\nWrote", nrow(lv_order), "significant LVs to", snakemake@output[["lv_sig"]], "\n")
